In [1]:
from __future__ import annotations

# =========================================================
# PS S6E4 - 046b LOG variant from family
#
# Purpose:
# - Build a LOG sibling of 046b / 024 family
# - Keep the 046b notebook skeleton as much as possible
# - Replace digit feature block with log-oriented feature block
# - Keep category remap, OrderedTE, XGB, and 3-stage bias refine
#
# Suggested experiment id:
#   exp_20260419_051_xgb046b_log_from_family
#
# Reference alignment:
# - 046b actual code uses 024 backbone + digit FE + category remap + OrderedTE
#   + XGB with best_params loaded from 046a + 3-stage bias refine. fileciteturn8file0
# - This notebook preserves that flow, but swaps only the feature expression family.
# =========================================================

In [2]:
import os
import json
import random
import warnings
from pathlib import Path
import itertools

import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import balanced_accuracy_score
import torch

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

In [3]:
# ---------------------------------------------------------
# CFG
# ---------------------------------------------------------
class CFG:
    EXP_ID = "exp_20260419_051_xgb046b_log_from_family"
    EXP_NAME = "xgb_log_orderedte_min_with_046b_params"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"
    SEED = 2026
    N_FOLDS = 5
    NUM_CLASSES = 3

    XGB_PARAMS = {
        "max_depth": 4,
        "colsample_bytree": 0.8,
        "subsample": 0.8,
        "n_estimators": 512,
        "learning_rate": 0.1,
        "early_stopping_rounds": 100,
        "random_state": SEED,
        "n_jobs": -1,
        "enable_categorical": True,
        "alpha": 5,
        "reg_lambda": 5,
        "max_leaves": 30,
        "min_child_weight": 2,
        "tree_method": "hist",
        "max_bin": 10000,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }

    best_params_payload = {}
    best_params_path = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/best_params.json"
    with open(best_params_path, mode="rt", encoding="utf-8") as f:
        best_params_payload = json.load(f)

    XGB_PARAMS.update(best_params_payload["best_params"])
    XGB_PARAMS["n_estimators"] = 2500

In [4]:
# ---------------------------------------------------------
# Utility
# ---------------------------------------------------------
def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)


def balanced_acc_score_mc(y_true, y_pred):
    if len(y_pred.shape) == 2:
        y_pred = np.argmax(y_pred, axis=1)
    c = len(np.unique(y_true))
    acc = 0.0
    for i in range(c):
        acc += np.sum((y_true == i) & (y_pred == i)) / np.sum(y_true == i) / c
    return acc


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def safe_logit_like(proba, eps=1e-12):
    return np.log(np.clip(proba, eps, 1.0))


def apply_bias(proba, bias):
    logp = safe_logit_like(proba)
    adjusted = logp + np.array(bias, dtype=np.float32).reshape(1, -1)
    adjusted = adjusted - adjusted.max(axis=1, keepdims=True)
    expv = np.exp(adjusted)
    out = expv / expv.sum(axis=1, keepdims=True)
    return out.astype(np.float32)


def score_bias(proba, y_true, bias):
    adj = apply_bias(proba, bias)
    pred = np.argmax(adj, axis=1)
    return balanced_accuracy_score(y_true, pred)


def run_grid_search(proba, y_true, bias_ranges):
    best_bias = None
    best_score = -1.0

    for b0, b1, b2 in itertools.product(*bias_ranges):
        bias = (b0, b1, b2)
        sc = score_bias(proba, y_true, bias)
        if sc > best_score:
            best_score = sc
            best_bias = bias

    return best_bias, float(best_score)


seed_everything(CFG.SEED)

In [5]:
# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
drop_cols = [CFG.ID_COL]

train = pd.read_csv(CFG.TRAIN_PATH).drop(drop_cols, axis=1)
test = pd.read_csv(CFG.TEST_PATH).drop(drop_cols, axis=1)

print(f"train.shape={train.shape}, test.shape={test.shape}")

target2idx = {v: i for i, v in enumerate(train[CFG.TARGET_COL].unique())}
idx2target = {v: k for k, v in target2idx.items()}

train[CFG.TARGET_COL] = train[CFG.TARGET_COL].map(target2idx)

CATS = [c for c in test.columns if train[c].dtype == object]
NUMS = [c for c in test.columns if c not in CATS]

print("CATS:", CATS)
print("NUMS:", NUMS)
print(train.head())

train.shape=(630000, 20), test.shape=(270000, 19)
CATS: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
NUMS: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
  Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  Sunlight_Hours  Wind_Speed_kmh  Crop_Type Crop_Growth_Stage  Season  \
0     Loamy     4.92          32.58            1.01                     3.05          15.01     50.61       725.99            5.90           16.79  Sugarcane            Sowing    Zaid   
1      Clay     7.08          56.61            0.44                     2.00          22.92     67.86       985.66            6.98            3.39      Wheat        Vegetative  Kharif   
2      Clay     5.69          27.71            0.81    

In [6]:
# ---------------------------------------------------------
# LOG feature block (swap for digit FE)
# ---------------------------------------------------------
# Keep 046b family shape idea: transform numerics, then remap category-like cols.
M = train[NUMS].max()


def signed_log1p(x: pd.Series) -> pd.Series:
    return np.sign(x) * np.log1p(np.abs(x))


def signed_sqrt(x: pd.Series) -> pd.Series:
    return np.sign(x) * np.sqrt(np.abs(x))


def add_log_features(df):
    df = df.copy()

    # preserve rounded raw numerics similar to family hygiene
    for c in NUMS:
        if M[c] < 10:
            df[c] = df[c].round(3)
        elif M[c] < 100:
            df[c] = df[c].round(2)
        else:
            df[c] = df[c].round(1)

    LOG_SOURCE = [
        "Soil_Moisture",
        "Organic_Carbon",
        "Electrical_Conductivity",
        "Rainfall_mm",
        "Sunlight_Hours",
        "Wind_Speed_kmh",
        "Field_Area_hectare",
        "Previous_Irrigation_mm",
    ]

    for c in LOG_SOURCE:
        x = df[c].astype(np.float32)
        df[f"{c}_log1p"] = np.log1p(np.clip(x, 0, None)).astype(np.float32)
        df[f"{c}_sqrt"] = np.sqrt(np.clip(x, 0, None)).astype(np.float32)
        df[f"{c}_is_zero"] = (np.abs(x) < 1e-12).astype("int8")
        df[f"{c}_diff_log1p"] = (x - np.log1p(np.clip(x, 0, None))).astype(np.float32)
        df[f"{c}_signed_log1p"] = signed_log1p(x).astype(np.float32)
        df[f"{c}_signed_sqrt"] = signed_sqrt(x).astype(np.float32)

        # coarse bins to create category-like structure for OrderedTE
        try:
            df[f"{c}_qbin"] = pd.qcut(x, q=12, duplicates="drop").astype(str)
        except Exception:
            df[f"{c}_qbin"] = x.astype(str)

    # light formula-like raw features, but do not drift too far from family
    df["soil_lt_25"] = (df["Soil_Moisture"] < 25).astype("int8")
    df["temp_gt_30"] = (df["Temperature_C"] > 30).astype("int8")
    df["rain_lt_300"] = (df["Rainfall_mm"] < 300).astype("int8")
    df["wind_gt_10"] = (df["Wind_Speed_kmh"] > 10).astype("int8")
    df["deotte_score_light"] = (
        df["soil_lt_25"] * 2
        + df["rain_lt_300"] * 2
        + df["temp_gt_30"]
        + df["wind_gt_10"]
        - (df["Crop_Growth_Stage"].eq("Harvest").astype("int8") * 2)
        - (df["Crop_Growth_Stage"].eq("Sowing").astype("int8") * 2)
        - (df["Mulching_Used"].eq("Yes").astype("int8"))
    )

    # selected pair categories only
    pair_defs = [
        ("Crop_Type", "Crop_Growth_Stage"),
        ("Irrigation_Type", "Water_Source"),
        ("Region", "Season"),
        ("Soil_Type", "Crop_Type"),
    ]
    for a, b in pair_defs:
        df[f"PAIR_{a}_{b}"] = df[a].astype(str) + "__" + df[b].astype(str)

    return df

train = add_log_features(train)
test = add_log_features(test)

DROP = [c for c in test.columns if test[c].nunique() == 1]
print("DROP:", DROP)

train.drop(DROP, axis=1, inplace=True)
test.drop(DROP, axis=1, inplace=True)

DROP: ['Soil_Moisture_is_zero', 'Organic_Carbon_is_zero', 'Electrical_Conductivity_is_zero', 'Rainfall_mm_is_zero', 'Sunlight_Hours_is_zero', 'Wind_Speed_kmh_is_zero', 'Field_Area_hectare_is_zero']


In [7]:
# ---------------------------------------------------------
# Category remap
# ---------------------------------------------------------
LOG_QBINS = [c for c in test.columns if c.endswith("_qbin")]
PAIRS = [c for c in test.columns if c.startswith("PAIR_")]
FLAGS = [
    "soil_lt_25", "temp_gt_30", "rain_lt_300", "wind_gt_10", "deotte_score_light"
]

CATEGORY = CATS + LOG_QBINS + PAIRS + FLAGS

for c in CATEGORY:
    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)

    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))

FEATURES = [c for c in train.columns if c != CFG.TARGET_COL]

print("n_CATEGORY =", len(CATEGORY))
print("n_FEATURES =", len(FEATURES))
print(train[FEATURES + [CFG.TARGET_COL]].head())

n_CATEGORY = 25
n_FEATURES = 77
   Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  Sunlight_Hours  Wind_Speed_kmh  Crop_Type  Crop_Growth_Stage  Season  \
0          2     4.92          32.58            1.01                     3.05          15.01     50.61        726.0            5.90           16.79          0                  3       2   
1          1     7.08          56.61            0.44                     2.00          22.92     67.86        985.7            6.98            3.39          4                  2       0   
2          1     5.69          27.71            0.81                     2.83          26.97     92.22       2201.7            6.05            3.85          1                  2       0   
3          0     5.65          13.32            1.33                     0.87          13.32     61.57       1357.3            9.12            2.31          4                  1       0   
4          1     7.96  

In [8]:
# ---------------------------------------------------------
# Class weight
# ---------------------------------------------------------
unique, counts = np.unique(train[CFG.TARGET_COL].values, return_counts=True)
count_dict = dict(zip(unique, counts))
avg_count = len(train) / len(unique)
weights_dict = {cls: avg_count / cnt for cls, cnt in count_dict.items()}
sample_weights = np.array([weights_dict[y] for y in train[CFG.TARGET_COL]])

print("weights_dict =", weights_dict)

weights_dict = {np.int64(0): np.float64(0.5676949153458749), np.int64(1): np.float64(0.8783891180136694), np.int64(2): np.float64(9.995716121662145)}


In [9]:
# ---------------------------------------------------------
# OrderedTE class (same as 046b)
# ---------------------------------------------------------
class OrderedTE:
    def __init__(self, a=1):
        self.a = a

    def fit(self, train, category_cols=None, target_col="target"):
        if category_cols is None:
            category_cols = []

        self.train = train.copy()
        self.target_col = target_col
        self.category_cols = category_cols

        self.classes_ = sorted(train[target_col].unique())
        self.num_classes_ = len(self.classes_)
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values

        for c in self.category_cols:
            stats_list = []

            for k, cls in enumerate(self.classes_):
                y_binary = (train[target_col] == cls).astype(int)

                df = train[[c]].copy()
                df["y"] = y_binary.values
                df["cnt"] = 1

                df["cum_cnt"] = df.groupby(c)["cnt"].cumsum() - df["cnt"]
                df["cum_sum"] = df.groupby(c)["y"].cumsum() - df["y"]

                smooth_prior = self.a * self.global_prior_[k]

                te_col = f"{c}_TE_cls{cls}"
                df[te_col] = (df["cum_sum"] + smooth_prior) / (df["cum_cnt"] + self.a)
                df.loc[df["cum_cnt"] == 0, te_col] = self.global_prior_[k]

                self.train[te_col] = df[te_col].values

                stats_df = df.groupby(c)["y"].agg(["count", "sum"]).reset_index()
                stats_df.columns = [c, f"{c}_count_cls{cls}", f"{c}_sum_cls{cls}"]
                stats_df[f"{c}_prior_cls{cls}"] = self.global_prior_[k]
                stats_list.append(stats_df)

            combined_stats = stats_list[0]
            for i in range(1, len(stats_list)):
                combined_stats = combined_stats.merge(stats_list[i], on=c, how="outer")
            setattr(self, f"{c}_stats", combined_stats)

        return self.train

    def transform(self, test):
        test = test.copy()

        for c in self.category_cols:
            stats_df = getattr(self, f"{c}_stats")
            test = test.merge(stats_df, on=c, how="left")

            for k, cls in enumerate(self.classes_):
                te_col = f"{c}_TE_cls{cls}"
                count_col = f"{c}_count_cls{cls}"
                sum_col = f"{c}_sum_cls{cls}"
                prior_col = f"{c}_prior_cls{cls}"

                if count_col in test.columns:
                    test[te_col] = (test[sum_col] + self.a * test[prior_col]) / (test[count_col] + self.a)
                    test[te_col] = test[te_col].fillna(test[prior_col])
                    test.drop([count_col, sum_col, prior_col], axis=1, inplace=True)
                else:
                    test[te_col] = self.global_prior_[k]

        return test

In [10]:
# ---------------------------------------------------------
# CV main
# ---------------------------------------------------------
X = train.drop([CFG.TARGET_COL], axis=1)
y = train[CFG.TARGET_COL]
test_X = test.copy()

oof_raw = np.zeros((len(y), CFG.NUM_CLASSES), dtype=np.float32)
pred_raw = np.zeros((len(test_X), CFG.NUM_CLASSES), dtype=np.float32)

kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=42)
fold_scores = []
feature_importance_rows = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    print(f"\nFold {fold}/{CFG.N_FOLDS}")

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()
    train_weights = sample_weights[train_idx]

    te = OrderedTE(a=1)
    full_df = pd.concat((X_train, y_train), axis=1)
    full_df["weight"] = train_weights

    te_train = te.fit(full_df.sample(frac=1, random_state=42), category_cols=FEATURES, target_col=CFG.TARGET_COL)

    X_train = te_train.drop([CFG.TARGET_COL, "weight"], axis=1)
    y_train = te_train[CFG.TARGET_COL]
    train_weights = te_train["weight"].values

    X_val = te.transform(X_val)
    X_test = te.transform(test_X)

    # drop original raw categorical columns only, keep remapped/binned/pair columns and TE outputs
    X_train.drop(CATS, axis=1, inplace=True)
    X_val.drop(CATS, axis=1, inplace=True)
    X_test.drop(CATS, axis=1, inplace=True)

    params = CFG.XGB_PARAMS.copy()
    if not torch.cuda.is_available():
        params["device"] = "cpu"

    model = XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        sample_weight=train_weights,
        verbose=100,
    )

    y_pred = model.predict_proba(X_val)
    oof_raw[val_idx] = y_pred
    pred_raw += model.predict_proba(X_test) / CFG.N_FOLDS

    fold_ba = balanced_acc_score_mc(y_val.values, y_pred)
    fold_scores.append(float(fold_ba))
    print(f"fold_ba = {fold_ba:.6f}")

    gain_dict = model.get_booster().get_score(importance_type="gain")
    fi = pd.DataFrame({"feature": X_train.columns})
    fi[f"gain_f{fold:02d}"] = fi["feature"].map(gain_dict).fillna(0.0)
    feature_importance_rows.append(fi)

# ---------------------------------------------------------
# raw CV
# ---------------------------------------------------------
raw_cv_ba = balanced_acc_score_mc(y.values, oof_raw)
print(f"raw_cv_ba = {raw_cv_ba:.6f}")


Fold 1/5
[0]	validation_0-mlogloss:1.01540
[100]	validation_0-mlogloss:0.07186
[200]	validation_0-mlogloss:0.06222
[300]	validation_0-mlogloss:0.05876
[400]	validation_0-mlogloss:0.05664
[500]	validation_0-mlogloss:0.05537
[600]	validation_0-mlogloss:0.05405
[700]	validation_0-mlogloss:0.05327
[800]	validation_0-mlogloss:0.05262
[900]	validation_0-mlogloss:0.05194
[1000]	validation_0-mlogloss:0.05153
[1100]	validation_0-mlogloss:0.05112
[1200]	validation_0-mlogloss:0.05075
[1300]	validation_0-mlogloss:0.05034
[1400]	validation_0-mlogloss:0.05005
[1500]	validation_0-mlogloss:0.04975
[1600]	validation_0-mlogloss:0.04952
[1700]	validation_0-mlogloss:0.04918
[1800]	validation_0-mlogloss:0.04891
[1900]	validation_0-mlogloss:0.04873
[2000]	validation_0-mlogloss:0.04846
[2100]	validation_0-mlogloss:0.04832
[2200]	validation_0-mlogloss:0.04819
[2300]	validation_0-mlogloss:0.04804
[2400]	validation_0-mlogloss:0.04798
[2499]	validation_0-mlogloss:0.04787
fold_ba = 0.975927

Fold 2/5
[0]	validat

In [11]:
# ---------------------------------------------------------
# bias search
# ---------------------------------------------------------
# Stage 1: coarse
coarse_vals = np.arange(-1.0, 1.01, 0.1)
best_bias_1, best_score_1 = run_grid_search(
    oof_raw, y,
    [coarse_vals, coarse_vals, coarse_vals]
)
print("stage1 best_bias:", best_bias_1, "score:", best_score_1)

# Stage 2: local refine
b0, b1, b2 = best_bias_1
local_vals_0 = np.arange(b0 - 0.10, b0 + 0.1001, 0.02)
local_vals_1 = np.arange(b1 - 0.10, b1 + 0.1001, 0.02)
local_vals_2 = np.arange(b2 - 0.10, b2 + 0.1001, 0.02)

best_bias_2, best_score_2 = run_grid_search(
    oof_raw, y,
    [local_vals_0, local_vals_1, local_vals_2]
)
print("stage2 best_bias:", best_bias_2, "score:", best_score_2)

# Stage 3: ultra-local refine
b0, b1, b2 = best_bias_2
ultra_vals_0 = np.arange(b0 - 0.02, b0 + 0.0201, 0.005)
ultra_vals_1 = np.arange(b1 - 0.02, b1 + 0.0201, 0.005)
ultra_vals_2 = np.arange(b2 - 0.02, b2 + 0.0201, 0.005)

best_bias_3, best_score_3 = run_grid_search(
    oof_raw, y,
    [ultra_vals_0, ultra_vals_1, ultra_vals_2]
)
print("stage3 best_bias:", best_bias_3, "score:", best_score_3)

final_bias = best_bias_3
final_cv = best_score_3

oof_biased = apply_bias(oof_raw, final_bias)
pred_biased = apply_bias(pred_raw, final_bias)

stage1 best_bias: (np.float64(-1.0), np.float64(-1.0), np.float64(-2.220446049250313e-16)) score: 0.9792618566499239
stage2 best_bias: (np.float64(-1.08), np.float64(-1.1), np.float64(-0.10000000000000023)) score: 0.9792659360263155
stage3 best_bias: (np.float64(-1.0850000000000004), np.float64(-1.1150000000000002), np.float64(-0.12000000000000023)) score: 0.9792740459536776


In [12]:
# ---------------------------------------------------------
# save outputs
# ---------------------------------------------------------
np.save(CFG.SAVE_DIR / "oof_xgb_log_orderedte_min_046bparams_biased_refined.npy", oof_biased)
np.save(CFG.SAVE_DIR / "pred_xgb_log_orderedte_min_046bparams_biased_refined.npy", pred_biased)
np.save(CFG.SAVE_DIR / "oof_xgb_log_orderedte_min_046bparams_raw.npy", oof_raw)
np.save(CFG.SAVE_DIR / "pred_xgb_log_orderedte_min_046bparams_raw.npy", pred_raw)
np.save(CFG.SAVE_DIR / "best_class_bias_refined.npy", np.array(final_bias, dtype=np.float32))

final_test_preds = np.argmax(pred_biased, axis=1)

submission = pd.read_csv(
    "/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")
submission[CFG.TARGET_COL] = pd.Series(final_test_preds).map(idx2target)
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

# feature importance aggregation
if len(feature_importance_rows) == 0:
    fi_merged = pd.DataFrame({"feature": FEATURES, "gain_mean": 0.0})
else:
    fi_merged = feature_importance_rows[0].copy()
    for item in feature_importance_rows[1:]:
        fi_merged = fi_merged.merge(item, on="feature", how="outer")
    gain_cols = [c for c in fi_merged.columns if c.startswith("gain_f")]
    fi_merged[gain_cols] = fi_merged[gain_cols].fillna(0.0)
    fi_merged["gain_mean"] = fi_merged[gain_cols].mean(axis=1)
    fi_merged = fi_merged.sort_values("gain_mean", ascending=False)
fi_merged.to_csv(CFG.SAVE_DIR / "feature_importance.csv", index=False)

pd.DataFrame({"feature": FEATURES}).to_csv(CFG.SAVE_DIR / "feature_columns.csv", index=False)

cv_result = {
    "exp_id": CFG.EXP_ID,
    "base_parent": "exp_20260416_046b_xgb_digit_orderedte_min_with_optuna_params",
    "variant": "log_from_family",
    "raw_cv": float(raw_cv_ba),
    "refined_biased_cv": float(final_cv),
    "best_bias": list(map(float, final_bias)),
    "fold_scores": fold_scores,
    "search": {
        "stage1": {"range": [-1.0, 1.0], "step": 0.1},
        "stage2": {"center": list(map(float, best_bias_1)), "width": 0.10, "step": 0.02},
        "stage3": {"center": list(map(float, best_bias_2)), "width": 0.02, "step": 0.005},
    },
    "notes": {
        "feature_pipeline": "046b family skeleton with LOG block replacing digit FE",
        "uses_046a_best_params_path": True,
        "ordered_te_in_fold": True,
        "category_remap": True,
        "digit_features_included": False,
        "log_features_included": True,
    },
}
save_json(cv_result, f"{CFG.SAVE_DIR}/cv_result.json")

print("final_bias:", final_bias)
print("final_cv:", final_cv)
print("saved to:", CFG.SAVE_DIR)

final_bias: (np.float64(-1.0850000000000004), np.float64(-1.1150000000000002), np.float64(-0.12000000000000023))
final_cv: 0.9792740459536776
saved to: /kaggle/working/exp_20260419_051_xgb046b_log_from_family
